# 08 — Explainable AI con SHAP

Este notebook explica el modelo de churn a nivel **global** e **individual** usando SHAP.

Para la explicabilidad se utiliza XGBoost porque su rendimiento es muy cercano al ensemble y `TreeSHAP` permite obtener explicaciones aditivas fieles al modelo. El ensemble de modelos heterogéneos se conserva como motor predictivo, pero no se fuerza una explicación SHAP incorrecta sobre un promedio de probabilidades.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path('..').resolve()
sys.path.append(str(ROOT))

from scripts.run_shap_explainability import run_shap_explainability

DATA_PATH = ROOT / 'data' / 'raw' / 'sales_marketing_customer_raw.csv'
df = pd.read_csv(DATA_PATH)
result = run_shap_explainability(df, save_outputs=True)

print('ROC-AUC:', round(result['summary']['roc_auc'], 4))
print('PR-AUC :', round(result['summary']['pr_auc'], 4))

## 1. Importancia global

In [ ]:
result['global_importance'].head(15)

La importancia global usa la media del valor absoluto de SHAP. Cuanto mayor es el valor, mayor es el impacto promedio de esa variable en las predicciones del modelo.

Las señales dominantes son `satisfaction_score`, `total_spent` y `support_tickets`, coherentes con el EDA y las pruebas de hipótesis.

## 2. Explicación individual

In [ ]:
result['summary'], result['local_explanation'].head(12)

Un valor SHAP positivo empuja la predicción hacia mayor riesgo de churn; un valor negativo la reduce.  
La explicación local permite justificar por qué un cliente particular fue clasificado como riesgoso sin convertir la predicción en una afirmación causal.

## 3. Controles de consistencia

In [ ]:
assert result['summary']['roc_auc'] > 0.91
assert result['summary']['pr_auc'] > 0.53
assert len(result['global_importance']) == result['prepared']['X_test_t'].shape[1]
print('Controles SHAP OK')

## Archivos generados

Al ejecutar el script se generan:

- `reports/figures/shap_summary.png`
- `reports/figures/shap_global_importance.png`
- gráficos de dependencia para satisfacción, gasto y tickets
- `reports/tables/shap_global_importance.csv`
- `reports/tables/shap_local_example.csv`

Estos elementos se reutilizarán luego en Streamlit y en el informe final.